# McDonald's Case Study: Exploratory Data Analysis

**Purpose:** record-keeping and re-testable EDA of the weekly sales and restaurant-facts data, following the seven-stage workflow in the EDA cheat sheet (context, structure, quality, distributions, relationships, patterns, synthesis).

**Mode:** descriptive and exploratory. Nothing here is confirmatory; every p-value is a flag for follow-up, not a result.

**How to use:** set `DATA_DIR` below (or the `MCD_DATA_DIR` environment variable), then *Restart & Run All*. Cells tagged **CHECK** are assertions: if the data extract changes and a documented property breaks, the notebook fails loudly instead of silently producing different numbers.

**Contents**
0. Context and questions
1. Structure and grain
2. Data quality (missingness, duplicates, impossible values, internal consistency, item hierarchy)
3. Single-variable distributions
4. Time structure (trend, seasonality, item mix)
5. Restaurant-level structure and relationships
6. Findings log, open questions, and regression checks

## 0. Context and questions

- **Source:** two extracts (weekly item sales; restaurant facts) plus a data dictionary supplied for a case study. Who collected them and why is not documented, so provenance is an **open question**.
- **Population:** 64 McDonald's restaurants in west Michigan (Kent, Kalamazoo, Ottawa, Grand Traverse counties). It is not a national sample, so claims stop at these restaurants.
- **Unit of analysis:** one row of sales = restaurant x menu item x week-ending Saturday.
- **Guiding questions:** What is in the data? How trustworthy is it? What is typical and what varies? What moves over time? How do restaurants differ, and which differences might be real?
- **What this data cannot answer:** causes (no promotions calendar, competitors, weather, or menu-change log), or anything outside these 64 restaurants.

In [ ]:
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib

# Force the in-notebook backend BEFORE importing pyplot. A GUI backend (Qt/Tk) can hang or
# crash a Jupyter kernel on the first plot. Falls back to non-interactive Agg outside Jupyter.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
from scipy import stats

try:
    display
except NameError:          # plain-python fallback
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
rng = np.random.default_rng(SEED)

DATA_DIR = Path(os.environ.get("MCD_DATA_DIR", "/mnt/user-data/uploads"))
SALES_PATH = DATA_DIR / "McD_Weekly_Sales.csv"
FACTS_PATH = DATA_DIR / "McD_Restaurant_Facts.csv"
DICT_PATH  = DATA_DIR / "McD_Data_Dictionary.xlsx"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# audit trail: record every exclusion / decision as we go
DECISIONS = []
def log(msg):
    DECISIONS.append(msg); print("LOG:", msg)

for p in (SALES_PATH, FACTS_PATH, DICT_PATH):
    print(p.name, p.exists(), f"{p.stat().st_size:,} bytes" if p.exists() else "")

### Environment and plotting smoke test
If the kernel dies here, the problem is the Python environment (matplotlib / numpy versions or backend), not the data or the analysis. Fix: run `pip install --upgrade numpy matplotlib ipykernel` in the same environment as the kernel, restart VS Code, and re-run.

In [ ]:
import sys
print("python    ", sys.version.split()[0])
print("numpy     ", np.__version__)
print("pandas    ", pd.__version__)
print("matplotlib", matplotlib.__version__, "| backend:", matplotlib.get_backend())

fig, ax = plt.subplots(figsize=(3, 2))
ax.plot([0, 1, 2], [0, 1, 4]); ax.set_title("smoke test")
plt.show(); plt.close(fig)
print("plotting works")

## 1. Structure and grain

In [ ]:
facts = pd.read_csv(FACTS_PATH, dtype={"Zip": str})
sales = pd.read_csv(SALES_PATH)

# Parse dates explicitly (never let the parser guess day-first vs month-first)
sales["d"] = pd.to_datetime(sales["wk_ending"], format="%m/%d/%y")

print("facts:", facts.shape, "| sales:", sales.shape)
display(sales.head())
display(sales.dtypes.to_frame("dtype").T)

In [ ]:
# Data dictionary as provided (note the blank descriptions)
try:
    dd = pd.read_excel(DICT_PATH, sheet_name=None)
    for name, t in dd.items():
        print("\n--", name); display(t)
except Exception as e:
    print("Could not read dictionary:", e)

In [ ]:
# CHECK: grain, keys, panel balance, join integrity
n_rest, n_item, n_week = sales.REST_KEY.nunique(), sales.ItemN.nunique(), sales.d.nunique()
print(n_rest, "restaurants x", n_item, "items x", n_week, "weeks =", n_rest*n_item*n_week, "| rows:", len(sales))

assert facts.REST_KEY.is_unique,                         "facts must have one row per restaurant"
assert not sales.duplicated(["REST_KEY","ItemN","d"]).any(), "sales grain (restaurant, item, week) must be unique"
assert len(sales) == n_rest*n_item*n_week,                "sales panel should be fully balanced"
assert set(sales.REST_KEY) == set(facts.REST_KEY),        "every restaurant must appear in both files"
assert (sales.groupby("ItemN").itemdesc.nunique() == 1).all(), "ItemN <-> itemdesc must be 1:1"

wk = np.sort(sales.d.unique())
gaps = pd.Series(wk).diff().dropna().value_counts()
print("week spacing:", gaps.to_dict(), "| weekday(s):", sales.d.dt.day_name().unique())
print("date range:", sales.d.min().date(), "to", sales.d.max().date())
assert (pd.Series(wk).diff().dropna() == pd.Timedelta(days=7)).all(), "weeks must be consecutive"

In [ ]:
# Item catalogue and item hierarchy (parent -> listed children)
items = sales[["ItemN","itemdesc"]].drop_duplicates().sort_values("ItemN").reset_index(drop=True)
HIERARCHY = {
    "Total All Day Breakfast Items": ["Total Breakfast Sandwich", "Hashbrown"],
    "Total Breakfast Sandwich": ["Total McMuffins", "Total Biscuits", "Total McGriddles", "Total Bagels"],
    "Total McMuffins": ["Egg McMuffin", "Sausage McMuffin", "Sausage Egg McMuffin"],
    "Total Biscuits": ["Bac Egg Ch Biscuit", "Sausage Biscuit", "Sausage Egg Biscuit"],
    "Total McGriddles": ["Bac Egg Ch McGriddle", "Sau Egg Ch McGriddle"],
    "Beef Category": ["Large Burgers", "Hamburger", "Cheeseburger", "McDouble"],
    "Large Burgers": ["Big Mac", "Core QP and QPC"],
    "Chicken Category": ["McChicken"],
}
display(items.T)
print("\nGrain statement: one row = one restaurant x one menu item x one week-ending Saturday.")
print("agc (transactions/day) is a restaurant-week quantity repeated on each of the 25 item rows.")
print("Items overlap (parents include children), so NEVER sum urws across itemdesc.")

## 2. Data quality

### 2a. Missingness

In [ ]:
meas = ["urws","wavg_price","upt","agc","adus","totunits"]
print("share missing per column:"); display(sales[meas].isna().mean().round(4).to_frame("missing"))

# CHECK: the six measures are missing together
assert (sales[meas].isna().nunique(axis=1) == 1).all(), "measures should be missing together"
sales["miss"] = sales.urws.isna()
print("missing rows:", sales.miss.sum(), f"({sales.miss.mean():.2%})")

In [ ]:
# Classify missingness. Whole restaurant-week blank = not trading / not open; 8-of-25 items = structural menu gap.
rw_miss = sales.groupby(["REST_KEY","d"]).miss.sum().rename("n_missing").reset_index()
print("items missing per restaurant-week:", rw_miss.n_missing.value_counts().sort_index().to_dict())

whole = rw_miss[rw_miss.n_missing == 25]
structural = rw_miss[rw_miss.n_missing == 8]

summary = []
for k, g in whole.groupby("REST_KEY"):
    summary.append((k, "whole weeks blank", len(g), g.d.min().date(), g.d.max().date()))
for k, g in structural.groupby("REST_KEY"):
    summary.append((k, "8 items never sold", len(g), g.d.min().date(), g.d.max().date()))
miss_tbl = pd.DataFrame(summary, columns=["REST_KEY","type","weeks","first","last"])
display(miss_tbl)

missing_items = sales[(sales.REST_KEY == structural.REST_KEY.iloc[0]) & sales.miss].itemdesc.unique()
print("\nStructurally missing items:", list(missing_items))
assert set(rw_miss.n_missing.unique()) <= {0, 8, 25}, "unexpected missingness pattern"

In [ ]:
# Missingness heatmap: restaurant x week (fraction of items missing)
m = rw_miss.pivot(index="REST_KEY", columns="d", values="n_missing") / 25
order = m.sum(axis=1).sort_values(ascending=False).index
arr = m.loc[order].to_numpy(dtype="float32")          # plain 64 x 189 array
fig, ax = plt.subplots(figsize=(10, 4.5))
im = ax.imshow(arr, aspect="auto", cmap="Reds", vmin=0, vmax=1, interpolation="nearest")
ax.set_xlabel("week index (0 = 2013-01-12)"); ax.set_ylabel("restaurant (most-missing first)")
ax.set_title("Missingness is structural: store openings, short closures, and one SPOD menu gap")
fig.colorbar(im, ax=ax, label="share of 25 items missing")
fig.tight_layout(); plt.show(); plt.close(fig)
log("Missing values are not random: openings (36277, 36302), closures (2308, 5238, 4635), structural menu gap (16773). No imputation.")

### 2b. Duplicates, impossible values, and internal consistency

In [ ]:
x = sales.dropna(subset=["urws"]).copy()

# Impossible / placeholder values
print("min urws:", x.urws.min(), "| zeros:", (x.urws == 0).sum(), "| negatives:", (x.urws < 0).sum())
print("price range:", x.wavg_price.min(), "-", x.wavg_price.max(), "| price <= 0:", (x.wavg_price <= 0).sum())
print("agc range:", x.agc.min(), "-", x.agc.max())
assert (x.urws > 0).all() and (x.wavg_price > 0).all() and (x.agc > 0).all()
assert x.urws.eq(x.totunits).all(), "totunits duplicates urws"
print("totunits == urws in every non-missing row -> totunits is redundant")

In [ ]:
# Days traded in the week, implied by adus = urws / days
x["days_open"] = (x.urws / x.adus).round(0).clip(1, 7)
print("implied days open per row:"); display(x.days_open.value_counts().sort_index().to_frame("rows").T)
print(f"rows with < 7 days: {(x.days_open < 7).mean():.2%}")

# Check upt formula: upt = urws / (agc * days) * 1000
upt_chk = x.urws / (x.agc * x.days_open) * 1000
print("upt formula max rel. error (full weeks):", float(((upt_chk - x.upt).abs() / x.upt)[x.days_open == 7].quantile(.99)))

# Short weeks by month: when do they cluster?
short = x[x.days_open < 7].drop_duplicates(["REST_KEY","d"])
display(short.groupby(short.d.dt.to_period("M")).size().sort_values(ascending=False).head(8).to_frame("restaurant-weeks"))
FULL = x[x.days_open == 7].copy()
log(f"Per-week volume analyses use full 7-day weeks only: kept {len(FULL):,} of {len(x):,} non-missing rows.")

In [ ]:
# agc consistency inside a restaurant-week (should be one value)
n_agc = x.groupby(["REST_KEY","d"]).agc.nunique()
print("restaurant-weeks with >1 distinct agc:", int((n_agc > 1).sum()), f"({(n_agc > 1).mean():.2%})")
RW = (x.groupby(["REST_KEY","d"]).agg(agc=("agc","median"), days=("days_open","max")).reset_index())

In [ ]:
# Item hierarchy: parents should be >= sum of listed children
P = x.pivot_table(index=["REST_KEY","d"], columns="itemdesc", values="urws")
rows = []
for par, ch in HIERARCHY.items():
    s_ch = P[ch].sum(axis=1, min_count=len(ch))
    ok = (P[par] >= s_ch - 1e-9)
    rows.append((par, len(ch), ok.mean(), ((P[par]-s_ch)/P[par]).median()))
hier = pd.DataFrame(rows, columns=["parent","n_children","share_parent>=children","median_unlisted_share"])
display(hier.round(3))

# CHECK: Breakfast Sandwich is exactly the sum of its four listed children
bs = P["Total Breakfast Sandwich"] - P[HIERARCHY["Total Breakfast Sandwich"]].sum(axis=1)
print("max |Breakfast Sandwich - children|:", bs.abs().max())

In [ ]:
# Price sanity: heavy tails by item
pr = FULL.groupby("itemdesc").wavg_price.describe(percentiles=[.01,.5,.99]).round(2)[["min","1%","50%","99%","max"]]
display(pr.sort_values("max", ascending=False).head(8))

# One suspicious high price (Sausage Egg Biscuit)
d1 = FULL[(FULL.itemdesc == "Sausage Egg Biscuit")].sort_values("wavg_price", ascending=False).head(3)
display(d1[["REST_KEY","d","urws","wavg_price"]])

# Promotional-looking lows: Big Mac priced < 60% of its median
bm = FULL[FULL.itemdesc == "Big Mac"]
low = bm[bm.wavg_price < 0.6*bm.wavg_price.median()]
print("Big Mac rows priced < 60% of median:", len(low), "| by year:", low.d.dt.year.value_counts().sort_index().to_dict())
log("Low Big Mac / Quarter Pounder prices concentrate in 2013 to early 2014: treated as promo/mix, not errors. One $5.53 Sausage Egg Biscuit price flagged, not removed.")

### 2c. Facts file

In [ ]:
print("missing per column:"); display(facts.isna().sum()[facts.isna().sum() > 0].to_frame("n_missing"))
display(facts[facts.incomeq_label.isna()][["REST_KEY","rest_label","City"]])

# CHECK: label and key agree; Zip stored as text (ZIP+4 variants exist)
assert (facts.rest_label.str.extract(r"^(\d+)")[0].astype(int) == facts.REST_KEY).all()
print("Zip lengths:", facts.Zip.str.len().value_counts().to_dict())
facts["zip5"] = facts.Zip.str[:5]

# Which restaurants are SPOD / non-freestanding?
display(facts[(facts.trad_label == "N") | (facts.REST_TYPE != "FREESTANDING")][["REST_KEY","rest_label","REST_TYPE","subtype_label"]])

# Coordinates plausibility
print(facts[["latitude","longitude"]].describe().loc[["min","max"]])
assert facts.latitude.between(41, 46).all() and facts.longitude.between(-88, -84).all()

# Category label consistency
for c in ["REST_PLYPL_TYP","REST_DRV_THRU_TYP","REST_TYPE","owner_label","urban_label","incomeq_label"]:
    print(f"\n{c}:", facts[c].value_counts(dropna=False).to_dict())

## 3. Single-variable distributions

In [ ]:
p = [.01,.05,.25,.5,.75,.95,.99]
print("POOLED urws is a mix of 25 different items, so the pooled mean is a rumor:")
display(FULL.urws.describe(percentiles=p).round(1).to_frame("urws").T)

by_item = FULL.groupby("itemdesc").urws.describe(percentiles=p).round(1)
by_item["skew"] = FULL.groupby("itemdesc").urws.skew().round(2)
display(by_item.sort_values("mean", ascending=False)[["mean","25%","50%","75%","99%","max","skew"]])

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, it in zip(axes.ravel(), ["Total Fries","Big Mac","McDouble","Hashbrown","Egg McMuffin","Sausage Egg Biscuit"]):
    FULL[FULL.itemdesc == it].urws.plot.hist(bins=50, ax=ax, color="#8b0000", alpha=.85)
    ax.set_title(it); ax.set_xlabel("units per full week")
plt.suptitle("Weekly units per restaurant: within-item shapes are mild right-skew, not pooled-mix shapes", y=1.01)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
RWf = RW[RW.days == 7]
RWf.agc.plot.hist(bins=60, ax=ax[0], color="#8b0000"); ax[0].set_title("Transactions per day (restaurant-weeks)")
ax[0].set_xlabel("agc")
RWf.groupby("REST_KEY").agc.mean().plot.hist(bins=20, ax=ax[1], color="#555"); ax[1].set_title("Mean daily transactions per restaurant")
ax[1].set_xlabel("mean agc"); plt.tight_layout(); plt.show()
print("agc percentiles:", RWf.agc.quantile(p).round(0).to_dict())

In [ ]:
# Mass at the extremes / spikes: price points are discrete (e.g. $1.00 dollar-menu floor)
for it in ["Hamburger","Sausage McMuffin","McChicken","Cheeseburger"]:
    v = FULL[FULL.itemdesc == it].wavg_price
    top = v.round(2).value_counts(normalize=True).head(3).round(3).to_dict()
    print(f"{it:18s} most common prices: {top}")

## 4. Time structure

In [ ]:
# Network average daily transactions per restaurant, by week
net = RWf.groupby("d").agc.mean()
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(net.index, net.values, lw=.8, color="#bbb", label="weekly mean")
ax.plot(net.index, net.rolling(13, center=True).mean(), lw=2, color="#8b0000", label="13-week rolling mean")
ax.set_ylabel("transactions per day (mean per restaurant)"); ax.set_title("Traffic: seasonal cycle on top of a steady decline")
ax.legend(); plt.tight_layout(); plt.show()

yr = RWf.assign(y=RWf.d.dt.year).groupby("y").agc.agg(["mean","median","count"]).round(0)
display(yr)

In [ ]:
# Seasonality: month index vs the overall mean (detrend first so decline does not leak in)
s = net.copy()
trend = s.rolling(52, center=True, min_periods=26).mean()
seas = (s / trend).groupby(s.index.month).mean()
print("seasonal index by month:", seas.round(3).to_dict())

try:
    from statsmodels.tsa.seasonal import STL
    res = STL(net.reset_index(drop=True), period=52, robust=True).fit()
    fig = res.plot(); fig.set_size_inches(10, 7); plt.tight_layout(); plt.show()
except Exception as e:
    print("statsmodels not available -> manual ratio-to-trend index used instead:", type(e).__name__)
    fig, ax = plt.subplots(figsize=(7, 3)); seas.plot.bar(ax=ax, color="#8b0000"); ax.axhline(1, c="k", lw=.6)
    ax.set_title("Seasonal index (ratio to 52-week trend)"); plt.tight_layout(); plt.show()

In [ ]:
# Season-neutral decline: each restaurant's first 52 weeks vs its last 52 weeks
first_cut = RWf.d.min() + pd.Timedelta(weeks=51)
last_cut  = RWf.d.max() - pd.Timedelta(weeks=51)
a = RWf[RWf.d <= first_cut].groupby("REST_KEY").agc.mean()
b = RWf[RWf.d >= last_cut].groupby("REST_KEY").agc.mean()
chg = (b / a - 1).dropna()
print("restaurants compared:", len(chg), "| median change:", f"{chg.median():.1%}", "| share declining:", f"{(chg < 0).mean():.1%}")
fig, ax = plt.subplots(figsize=(6.5, 3.2)); chg.plot.hist(bins=20, ax=ax, color="#8b0000")
ax.axvline(0, c="k", lw=.8); ax.set_xlabel("change in daily transactions, first 52 wks to last 52 wks")
ax.set_title("Almost every restaurant lost traffic"); plt.tight_layout(); plt.show()

In [ ]:
# Item mix over time (units per full restaurant-week). 2016 covers Jan-Aug only: directional.
FULL["year"] = FULL.d.dt.year
focus = ["Total All Day Breakfast Items","Total McMuffins","Beef Category","Chicken Category","Total Fries","McDouble","Big Mac","Hashbrown"]
U = FULL[FULL.itemdesc.isin(focus)].pivot_table(index="year", columns="itemdesc", values="urws").round(0)
display(U.T)
idx = (U / U.iloc[0] * 100).round(0)
fig, ax = plt.subplots(figsize=(9, 4))
for c in idx.columns: ax.plot(idx.index, idx[c], marker="o", label=c)
ax.axhline(100, c="k", lw=.6); ax.set_xticks(idx.index); ax.set_ylabel("index (2013 = 100)")
ax.set_title("Units per restaurant-week by item, indexed"); ax.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

In [ ]:
# McDouble: price vs units by quarter (raw units are not per-transaction; compare upt too)
q = FULL[FULL.itemdesc == "McDouble"].assign(q=lambda t: t.d.dt.to_period("Q")).groupby("q").agg(
        units=("urws","mean"), price=("wavg_price","mean"), per_1000_txn=("upt","mean")).round(2)
fig, ax = plt.subplots(figsize=(9, 3.5)); ax2 = ax.twinx()
ax.plot(q.index.astype(str), q.units, color="#8b0000", marker="o"); ax2.plot(q.index.astype(str), q.price, color="#1f77b4", marker="s")
ax.set_ylabel("units / week", color="#8b0000"); ax2.set_ylabel("avg price ($)", color="#1f77b4")
ax.set_title("McDouble: price up ~40%, units down (association only)"); plt.setp(ax.get_xticklabels(), rotation=60, fontsize=7)
plt.tight_layout(); plt.show()

## 5. Restaurant-level structure and relationships

In [ ]:
# One row per restaurant: mean daily transactions (full weeks) joined to facts
R = (RWf.groupby("REST_KEY").agc.agg(mean_agc="mean", sd_agc="std", weeks="count").round(1)
        .join(facts.set_index("REST_KEY")))
R["mean_agc"].describe().round(0).to_frame("mean_agc").T

order = R.sort_values("mean_agc")
fig, ax = plt.subplots(figsize=(7, 9))
colors = order.owner_label.map({"MCOPCO":"#8b0000","LICENSEE":"#999"})
ax.barh(order.index.astype(str), order.mean_agc, color=colors)
ax.set_xlabel("mean daily transactions"); ax.set_title("Restaurant ranking (red = company-operated)")
plt.tight_layout(); plt.show()
display(R.sort_values("mean_agc").head(3)[["City","REST_TYPE","subtype_label","mean_agc"]])

In [ ]:
def group_table(col):
    g = R.groupby(col, dropna=False).mean_agc.agg(["count","mean","median"]).round(0)
    return g.sort_values("median", ascending=False)

for c in ["owner_label","REST_DRV_THRU_TYP","REST_PLYPL_TYP","urban_label","incomeq_label","County","REST_TYPE"]:
    print("\n" + c); display(group_table(c))

In [ ]:
# Ownership x drive-thru: are the two differences separable?
display(pd.crosstab(R.owner_label, R.REST_DRV_THRU_TYP))
R["sbs"] = np.where(R.REST_DRV_THRU_TYP == "SIDE BY SIDE 2 BOOTH", "side-by-side", "other")
display(R.groupby(["owner_label","sbs"]).mean_agc.agg(["count","median"]).round(0))

# Tests (exploratory; report effect size and n, not just p)
lic = R[R.owner_label == "LICENSEE"].mean_agc; co = R[R.owner_label == "MCOPCO"].mean_agc
u = stats.mannwhitneyu(co, lic)
print(f"Company vs licensee: medians {co.median():.0f} vs {lic.median():.0f}; MWU p={u.pvalue:.4f}; n={len(co)}/{len(lic)}")
sb = R[R.sbs == "side-by-side"].mean_agc; ot = R[R.sbs == "other"].mean_agc
u2 = stats.mannwhitneyu(sb, ot)
print(f"Side-by-side vs other: medians {sb.median():.0f} vs {ot.median():.0f}; MWU p={u2.pvalue:.4f}; n={len(sb)}/{len(ot)}")

inc = R.dropna(subset=["incomeq_label"])
k = stats.kruskal(*[g.mean_agc.values for _, g in inc.groupby("incomeq_label")])
print(f"Income quintile (Kruskal): p={k.pvalue:.3f}  (no clear gradient; top group n={int((inc.incomeq_label=='1-Highest Income').sum())})")
rho_lat = stats.spearmanr(R.mean_agc, R.latitude)
print(f"Latitude vs mean_agc: rho={rho_lat.statistic:.2f}, p={rho_lat.pvalue:.3f}  (one of many looks: a flag, not a result)")

# Bootstrap interval for company-minus-licensee difference in medians
def boot_diff(a, b, n=5000):
    d = [np.median(rng.choice(a, len(a))) - np.median(rng.choice(b, len(b))) for _ in range(n)]
    return np.percentile(d, [2.5, 97.5])
print("95% bootstrap CI, difference in medians (company - licensee):", boot_diff(co.values, lic.values).round(0))

In [ ]:
# Spatial view: raw points, size/colour = mean daily transactions
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(R.longitude, R.latitude, c=R.mean_agc, s=R.mean_agc/12, cmap="Reds", edgecolor="k", linewidth=.4)
plt.colorbar(sc, label="mean daily transactions"); ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title("Where the restaurants are (Grand Traverse cluster is far north)"); plt.tight_layout(); plt.show()

In [ ]:
# Categorical x categorical: demographic fields are partly redundant (urban type vs income vs life stage)
from itertools import combinations
def cramers_v(a, b):
    t = pd.crosstab(a, b); chi2 = stats.chi2_contingency(t)[0]; n = t.values.sum()
    return np.sqrt(chi2 / (n * (min(t.shape) - 1)))
cat = ["owner_label","REST_DRV_THRU_TYP","urban_label","incomeq_label","social_label","lstage_label","County"]
cv = pd.DataFrame(index=cat, columns=cat, dtype=float)
for a_, b_ in combinations(cat, 2):
    cv.loc[a_, b_] = cv.loc[b_, a_] = cramers_v(R[a_].fillna("NA"), R[b_].fillna("NA"))
for c_ in cat: cv.loc[c_, c_] = 1.0
display(cv.round(2))
print("Caution: cells are sparse with n=64; Cramer's V is biased upward here.")

## 6. Findings log, open questions, and regression checks

### Findings (each: observation, evidence, confidence, caveat)

| # | Observation | Evidence | Confidence |
|---|---|---|---|
| F1 | Panel is complete: 64 restaurants x 25 items x 189 consecutive weeks (2013-01-12 to 2016-08-20), unique grain | section 1 checks | High |
| F2 | 1.6% of rows are missing, all structural (2 openings, 3 closures, 1 SPOD without 8 items) | section 2a | High |
| F3 | About 3% of rows are short trading weeks (mostly 6 days; peak Nov-Dec 2013) | section 2b | High (days inferred) |
| F4 | Item rows overlap; summing across items double counts | section 2b hierarchy | High |
| F5 | Daily transactions fell ~12% (median per restaurant, first vs last 52 weeks); nearly all restaurants declined | section 4 | High on the fact; cause unknown |
| F6 | Strong seasonality: January lowest, May to October highest | section 4 | High |
| F7 | Mix shift: breakfast units up, beef/chicken/fries down; McDouble price up ~40% with units down | section 4 | Medium (2016 partial; association only) |
| F8 | Company-operated restaurants and side-by-side drive-thrus both average more daily transactions. Among licensees alone, side-by-side (median ~1,420) beats other layouts (~1,120); among side-by-side restaurants the ownership gap is much smaller (~1,540 vs ~1,420) | section 5 | Low-medium: overlapping, n=64, only 1 company restaurant is non-side-by-side |
| F9 | No clear income-quintile gradient | section 5 | Low (small groups) |

### Open questions
1. Provenance: who produced the extract, and why these 64 restaurants?
2. What explains 2308 and 5238's 10-week gaps (remodel? closure?) and the 8 items 16773 never sells?
3. What drove the McDouble repricing and the 2013 low Big Mac prices (promotions)?
4. Is the traffic decline market-wide or specific to this operator? Nothing here separates the two.
5. Are ownership and drive-thru type separable effects, or one effect seen twice?
6. Meaning and currency of `wavg_price`; definitions of play place, drive-thru, owner, and demographic labels (blank in dictionary).

In [ ]:
# REGRESSION CHECKS: documented findings, re-tested on every run
assert len(sales) == 302_400
assert sales.miss.sum() == 4_762
assert set(whole.REST_KEY) == {2308, 4635, 5238, 36277, 36302}
assert set(structural.REST_KEY) == {16773}
assert 0.02 < (x.days_open < 7).mean() < 0.05
assert (chg < 0).mean() > 0.9, "traffic decline should hold for >90% of restaurants"
assert -0.2 < chg.median() < -0.05
assert co.median() > lic.median(), "company-operated median should exceed licensee median"
print("All regression checks passed.")
print("\nDecision log:"); [print(" -", d) for d in DECISIONS]